# Online Course Registration and Student Enrollment Analysis System
### Velalar College of Engineering and Technology, Erode - 12
**Department of Information Technology | Class: 25IT3A**  
**Activity:** Project-Based Assessment (Scenario 17)  
**Author:** Dhanush .K (Register No: 732925ITR017) | Team: 17, 47, 77, 107  
**Course Code:** 25ITC31 - Foundations of Data Science  

---
### Objectives:
1. Retrieve real-time enrollment and student data from SQLite database.
2. Perform data cleaning, type conversions, and preprocessing using NumPy and Pandas.
3. Conduct Exploratory Data Analysis (EDA) on course popularity, completion rates, and seat utilization.
4. Generate high-quality statistical visualizations using Matplotlib and Seaborn.

## 1. Import Libraries and Connect to Database

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120

db_path = os.path.join("..", "database", "course_enrollment.db")
conn = sqlite3.connect(db_path)
print("Connected to SQLite Database at:", db_path)

## 2. Load Datasets into Pandas DataFrames (Unit 2 & 3)

In [ ]:
query = """
SELECT 
    e.enrollment_id,
    e.enrollment_date,
    e.status AS enrollment_status,
    s.student_id,
    s.name AS student_name,
    s.department,
    s.year_of_study,
    c.course_id,
    c.course_name,
    c.category,
    c.duration_weeks,
    c.max_capacity,
    COALESCE(pr.attendance_percentage, 0.0) AS attendance_percentage,
    pr.assessment_score,
    COALESCE(pr.completion_status, 'Incomplete') AS completion_status,
    pr.completion_date
FROM enrollments e
JOIN students s ON e.student_id = s.student_id
JOIN courses c ON e.course_id = c.course_id
LEFT JOIN participation_records pr ON e.enrollment_id = pr.enrollment_id;
"""

df = pd.read_sql_query(query, conn)
courses_df = pd.read_sql_query("SELECT * FROM courses", conn)
print("Dataset Shape:", df.shape)
df.head()

## 3. Data Cleaning and Transformation (Unit 4)
- Convert date columns to datetime objects.
- Impute missing assessment scores for incomplete courses.
- Add department acronyms for concise plotting.

In [ ]:
df['enrollment_date'] = pd.to_datetime(df['enrollment_date'])
df['completion_date'] = pd.to_datetime(df['completion_date'])

dept_map = {
    'Information Technology': 'IT',
    'Computer Science and Engineering': 'CSE',
    'Electronics and Communication Engineering': 'ECE',
    'Artificial Intelligence and Data Science': 'AI&DS',
    'Mechanical Engineering': 'MECH'
}
df['dept_code'] = df['department'].map(dept_map)

print("Missing Value Count:")
print(df.isnull().sum())

## 4. Exploratory Data Analysis & Visualizations (Unit 5)

### 4.1 Most Popular Courses by Enrollment

In [ ]:
plt.figure(figsize=(10, 5))
course_counts = df['course_name'].value_counts()
sns.barplot(x=course_counts.values, y=course_counts.index, hue=course_counts.index, palette='mako', legend=False)
plt.title('Course Popularity: Enrollments per Course', fontsize=14, fontweight='bold')
plt.xlabel('Enrolled Students Count')
plt.ylabel('Course Title')
plt.show()

### 4.2 Seat Capacity Utilization vs Active Enrollments

In [ ]:
cap_summary = courses_df.merge(df.groupby('course_id').size().reset_index(name='enrolled'), on='course_id', how='left').fillna(0)

x = np.arange(len(cap_summary))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, cap_summary['max_capacity'], width, label='Max Capacity', color='#3b528b')
plt.bar(x + width/2, cap_summary['enrolled'], width, label='Current Enrolled', color='#5ec962')
plt.xticks(x, cap_summary['course_id'])
plt.xlabel('Course ID', fontweight='bold')
plt.ylabel('Seats', fontweight='bold')
plt.title('Course Capacity Utilization Analysis', fontsize=14, fontweight='bold')
plt.legend()
plt.show()

### 4.3 Department-wise Student Participation (Donut Chart)

In [ ]:
dept_counts = df['dept_code'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(dept_counts, labels=dept_counts.index, autopct='%1.1f%%', startangle=140, pctdistance=0.8)
centre_circle = plt.Circle((0,0), 0.60, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Enrollment Distribution by Department', fontsize=14, fontweight='bold')
plt.show()

### 4.4 Course Category vs Duration Heatmap

In [ ]:
pivot = pd.pivot_table(df, values='enrollment_id', index='category', columns='duration_weeks', aggfunc='count', fill_value=0)
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, cmap='Blues', fmt='d', linewidths=0.5)
plt.title('Enrollment Density: Category vs Duration (Weeks)', fontsize=14, fontweight='bold')
plt.show()

## 5. Statistical Summary and Insights
- Analysis shows high student interest in **Full Stack Web Development** and **Applied Machine Learning**.
- Departmental participation is led by **Information Technology** and **Computer Science & Engineering**.
- Capacity limits safely prevented over-enrollment across all technical training modules.